# Credit Profile Tear Sheet

Build a borrower's quarterly model, compute a structured `credit_assessment`
(leverage, interest coverage, free cash flow, plus a per-period series), and
render a `credit_tearsheet`. Per-instrument coverage and covenant headroom are
deal terms supplied by the caller (or sourced from your credit engine).

In [ ]:
import datetime as dt

from finstack_quant import reporting
from finstack_quant.statements import Evaluator, ModelBuilder
from finstack_quant.statements_analytics import credit_assessment

qs = ["2024Q1", "2024Q2", "2024Q3", "2024Q4", "2025Q1", "2025Q2", "2025Q3", "2025Q4"]
b = ModelBuilder("borrower")
b.periods("2024Q1..2025Q4", "2024Q4")
b.value("revenue", list(zip(qs, [100, 104, 108, 112, 116, 120, 124, 128])))
b.value("cogs", list(zip(qs, [55, 57, 59, 61, 63, 65, 67, 69])))
b.compute("gross_profit", "revenue - cogs")
b.value("opex", list(zip(qs, [18, 18, 19, 19, 20, 20, 21, 21])))
b.compute("ebitda", "gross_profit - opex")
b.value("interest_expense", list(zip(qs, [4.0] * 8)))
b.value("total_debt", list(zip(qs, [300, 300, 295, 295, 290, 290, 285, 285])))
b.value("free_cash_flow", list(zip(qs, [8, 9, 10, 11, 12, 13, 14, 15])))
result = Evaluator().evaluate(b.build())

assessment = credit_assessment(result.to_json(), "2025Q4")
print("Leverage:", round(assessment["leverage_ratio"], 2), " Coverage:", round(assessment["interest_coverage"], 2))

## Credit profile

Leverage/coverage trend and free cash flow come from the structured
`credit_assessment`. Per-instrument coverage (DSCR / interest cover / LTV) and
covenant compliance are passed in as deal terms. (LTV is expressed in percent
units, e.g. `55.0` renders as `55.0%`.)

In [ ]:
coverage = [
    {"instrument": "Term Loan B", "dscr": 1.8, "interest_coverage": 3.2, "ltv": 55.0},
    {"instrument": "Senior Notes", "dscr": 2.1, "interest_coverage": 3.8, "ltv": 42.0},
]
covenants = [
    {"covenant": "Max Net Leverage", "threshold": 4.0, "current": 3.1, "headroom": 0.9, "status": "Pass"},
    {"covenant": "Min Interest Coverage", "threshold": 2.5, "current": 3.2, "headroom": 0.7, "status": "Pass"},
    {"covenant": "Min DSCR", "threshold": 1.5, "current": 1.4, "headroom": -0.1, "status": "Breach"},
]
reporting.credit_tearsheet(
    assessment,
    results=result,
    coverage=coverage,
    covenants=covenants,
    title="Acme Corp — Credit",
    generated=dt.date(2026, 6, 22),
)

## Saving a standalone HTML file

```python
ts = reporting.credit_tearsheet(assessment, results=result, generated=dt.date(2026, 6, 22))
ts.save("credit_tearsheet.html")
```